<a href="https://colab.research.google.com/github/amzad-786githumb/Privacy-Preserving-Synthetic-Tabular-Data-Generation-Using-Generative-Adversarial-Networks/blob/main/06_Privacy_Preserving_CTGAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [77]:
# =============================================================================
# Notebook 06
# Privacy-Preserving GAN for Synthetic Tabular Data
# Part 6.1 : Environment Setup
# Block 1 : Install Required Packages
# =============================================================================

print("=" * 80)
print("Installing Required Packages")
print("=" * 80)

# Upgrade pip
!pip -q install --upgrade pip

# Deep Learning
!pip -q install torch torchvision torchaudio

# Differential Privacy
!pip -q install opacus

# Data Processing
!pip -q install pandas numpy scipy scikit-learn

# Visualization
!pip -q install matplotlib seaborn

# Utilities
!pip -q install tqdm pyyaml joblib

print("\nAll packages installed successfully.")

Installing Required Packages

All packages installed successfully.


In [78]:
# =============================================================================
# Block 2 : Import Libraries
# =============================================================================

import os
import gc
import random
import warnings
from pathlib import Path

import yaml
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from scipy.stats import (
    ks_2samp,
    wasserstein_distance
)

from scipy.spatial.distance import jensenshannon

from opacus import PrivacyEngine

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [79]:
# =============================================================================
# Block 3 : Mount Google Drive
# =============================================================================

from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/SPP_GAN_Project"
)

assert PROJECT_DIR.exists(), \
    f"Project folder not found:\n{PROJECT_DIR}"

print("=" * 80)
print("Google Drive Mounted Successfully")
print("=" * 80)

print(PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive Mounted Successfully
/content/drive/MyDrive/SPP_GAN_Project


In [80]:
# =============================================================================
# Block 4 : GPU Configuration
# =============================================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 80)
print("Hardware Information")
print("=" * 80)

print(f"Device : {DEVICE}")

if torch.cuda.is_available():

    print(f"GPU Name : {torch.cuda.get_device_name(0)}")

    gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory
        /1024**3
    )

    print(f"GPU Memory : {gpu_memory:.2f} GB")

else:

    print("Running on CPU")

Hardware Information
Device : cpu
Running on CPU


In [81]:
# =============================================================================
# Block 5 : Random Seed
# =============================================================================

SEED = 42

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed(SEED)

    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True

torch.backends.cudnn.benchmark = False

print("=" * 80)
print("Random Seed Initialized")
print("=" * 80)

print(f"Seed : {SEED}")

Random Seed Initialized
Seed : 42


In [82]:
# =============================================================================
# Block 6 : Verify Environment
# =============================================================================

print("=" * 80)
print("Environment Summary")
print("=" * 80)

print(f"Python Version     : {os.sys.version.split()[0]}")

print(f"PyTorch Version    : {torch.__version__}")

print(f"CUDA Available     : {torch.cuda.is_available()}")

if torch.cuda.is_available():

    print(f"CUDA Version       : {torch.version.cuda}")

print(f"Working Directory  : {PROJECT_DIR}")

print(f"Training Device    : {DEVICE}")

print("\nEnvironment is ready for training.")

Environment Summary
Python Version     : 3.12.13
PyTorch Version    : 2.11.0+cpu
CUDA Available     : False
Working Directory  : /content/drive/MyDrive/SPP_GAN_Project
Training Device    : cpu

Environment is ready for training.


In [83]:
# =============================================================================
# Part 6.2 : Project Configuration
# Block 1 : Load Configuration
# =============================================================================

CONFIG_PATH = PROJECT_DIR / "config.yaml"

assert CONFIG_PATH.exists(), \
    f"Configuration file not found:\n{CONFIG_PATH}"

with open(CONFIG_PATH, "r") as file:
    config = yaml.safe_load(file)

print("=" * 80)
print("Configuration Loaded Successfully")
print("=" * 80)

for key, value in config.items():
    print(f"{key:<25}: {value}")

Configuration Loaded Successfully
batch_size               : 256
beta1                    : 0.5
beta2                    : 0.999
delta                    : 1e-05
device                   : cuda
epochs                   : 300
gradient_clip            : 1.0
latent_dimension         : 128
learning_rate            : 0.0002
noise_multiplier         : 1.1
optimizer                : Adam
privacy_budget           : 4
seed                     : 42


In [84]:
# =============================================================================
# Block 2 : Project Directories
# =============================================================================

DATA_DIR = PROJECT_DIR / "datasets"

RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
SYNTHETIC_DATA_DIR = DATA_DIR / "synthetic"

MODEL_DIR = PROJECT_DIR / "models"
RESULT_DIR = PROJECT_DIR / "results"
REPORT_DIR = PROJECT_DIR / "reports"
LOG_DIR = PROJECT_DIR / "logs"
METADATA_DIR = PROJECT_DIR / "metadata"

DIRECTORIES = {
    "Raw Data": RAW_DATA_DIR,
    "Processed Data": PROCESSED_DATA_DIR,
    "Synthetic Data": SYNTHETIC_DATA_DIR,
    "Models": MODEL_DIR,
    "Results": RESULT_DIR,
    "Reports": REPORT_DIR,
    "Logs": LOG_DIR,
    "Metadata": METADATA_DIR
}

print("=" * 80)
print("Creating Project Directories")
print("=" * 80)

for name, path in DIRECTORIES.items():

    path.mkdir(parents=True, exist_ok=True)

    print(f"{name:<20}: {path}")

print("\nProject directories are ready.")

Creating Project Directories
Raw Data            : /content/drive/MyDrive/SPP_GAN_Project/datasets/raw
Processed Data      : /content/drive/MyDrive/SPP_GAN_Project/datasets/processed
Synthetic Data      : /content/drive/MyDrive/SPP_GAN_Project/datasets/synthetic
Models              : /content/drive/MyDrive/SPP_GAN_Project/models
Results             : /content/drive/MyDrive/SPP_GAN_Project/results
Reports             : /content/drive/MyDrive/SPP_GAN_Project/reports
Logs                : /content/drive/MyDrive/SPP_GAN_Project/logs
Metadata            : /content/drive/MyDrive/SPP_GAN_Project/metadata

Project directories are ready.


In [85]:
# =============================================================================
# Block 3 : Load Processed Datasets
# =============================================================================

DATASET_FILES = {

    "Adult Income":
        "adult_income_processed.csv",

    "Bank Marketing":
        "bank_marketing_processed.csv",

    "Breast Cancer":
        "breast_cancer_processed.csv"

}

datasets = {}

print("=" * 80)
print("Loading Processed Datasets")
print("=" * 80)

for dataset_name, filename in DATASET_FILES.items():

    filepath = PROCESSED_DATA_DIR / filename

    assert filepath.exists(), \
        f"Dataset not found:\n{filepath}"

    df = pd.read_csv(filepath)

    datasets[dataset_name] = df

    print(f"{dataset_name:<20} Shape : {df.shape}")

print("\nAll datasets loaded successfully.")

Loading Processed Datasets
Adult Income         Shape : (32510, 15)
Bank Marketing       Shape : (45211, 17)
Breast Cancer        Shape : (569, 32)

All datasets loaded successfully.


In [86]:
# =============================================================================
# Block 4 : Dataset Verification
# =============================================================================

summary = []

for dataset_name, df in datasets.items():

    summary.append({

        "Dataset":
            dataset_name,

        "Rows":
            df.shape[0],

        "Columns":
            df.shape[1],

        "Missing Values":
            int(df.isnull().sum().sum()),

        "Duplicate Rows":
            int(df.duplicated().sum()),

        "Memory (MB)":
            round(
                df.memory_usage(deep=True).sum()/1024**2,
                2
            )

    })

summary = pd.DataFrame(summary)

print("=" * 80)
print("Dataset Summary")
print("=" * 80)

display(summary)

Dataset Summary


,Dataset,Rows,Columns,Missing Values,Duplicate Rows,Memory (MB)
0,Adult Income,32510,15,0,0,3.72
1,Bank Marketing,45211,17,0,0,5.86
2,Breast Cancer,569,32,0,0,0.14


In [87]:
# =============================================================================
# Block 5 : Experiment Parameters
# =============================================================================

SEED = config["seed"]

LATENT_DIM = config["latent_dimension"]

BATCH_SIZE = config["batch_size"]

EPOCHS = config["epochs"]

LEARNING_RATE = config["learning_rate"]

BETA1 = config["beta1"]

BETA2 = config["beta2"]

NOISE_MULTIPLIER = config["noise_multiplier"]

MAX_GRAD_NORM = config["gradient_clip"]

EPSILON = config["privacy_budget"]

DELTA = config["delta"]

print("=" * 80)
print("Experiment Parameters")
print("=" * 80)

print(f"Seed                : {SEED}")
print(f"Latent Dimension    : {LATENT_DIM}")
print(f"Batch Size          : {BATCH_SIZE}")
print(f"Epochs              : {EPOCHS}")
print(f"Learning Rate       : {LEARNING_RATE}")
print(f"Optimizer Beta1     : {BETA1}")
print(f"Optimizer Beta2     : {BETA2}")
print(f"Privacy Budget (ε)  : {EPSILON}")
print(f"Delta (δ)           : {DELTA}")
print(f"Noise Multiplier    : {NOISE_MULTIPLIER}")
print(f"Gradient Clip       : {MAX_GRAD_NORM}")

Experiment Parameters
Seed                : 42
Latent Dimension    : 128
Batch Size          : 256
Epochs              : 300
Learning Rate       : 0.0002
Optimizer Beta1     : 0.5
Optimizer Beta2     : 0.999
Privacy Budget (ε)  : 4
Delta (δ)           : 1e-05
Noise Multiplier    : 1.1
Gradient Clip       : 1.0


In [88]:
# =============================================================================
# Block 6 : Dataset Information
# =============================================================================

dataset_info = {}

for dataset_name, df in datasets.items():

    dataset_info[dataset_name] = {

        "rows": df.shape[0],

        "columns": df.shape[1],

        "feature_names": list(df.columns),

        "input_dimension": df.shape[1]

    }

joblib.dump(
    dataset_info,
    METADATA_DIR / "dataset_info.pkl"
)

print("=" * 80)
print("Dataset Information Saved")
print("=" * 80)

for dataset_name, info in dataset_info.items():

    print(f"{dataset_name}")

    print(f"Rows              : {info['rows']}")
    print(f"Input Features    : {info['input_dimension']}")
    print("-"*50)

Dataset Information Saved
Adult Income
Rows              : 32510
Input Features    : 15
--------------------------------------------------
Bank Marketing
Rows              : 45211
Input Features    : 17
--------------------------------------------------
Breast Cancer
Rows              : 569
Input Features    : 32
--------------------------------------------------


In [89]:
# =============================================================================
# Block 7 : Project Summary
# =============================================================================

print("=" * 80)
print("Project Configuration Summary")
print("=" * 80)

print(f"Project Directory     : {PROJECT_DIR}")

print(f"Datasets Loaded       : {len(datasets)}")

print(f"Training Device       : {DEVICE}")

print(f"Model Directory       : {MODEL_DIR}")

print(f"Results Directory     : {RESULT_DIR}")

print(f"Reports Directory     : {REPORT_DIR}")

print(f"Metadata Directory    : {METADATA_DIR}")

print("\nNotebook 06 Configuration Completed Successfully.")

Project Configuration Summary
Project Directory     : /content/drive/MyDrive/SPP_GAN_Project
Datasets Loaded       : 3
Training Device       : cpu
Model Directory       : /content/drive/MyDrive/SPP_GAN_Project/models
Results Directory     : /content/drive/MyDrive/SPP_GAN_Project/results
Reports Directory     : /content/drive/MyDrive/SPP_GAN_Project/reports
Metadata Directory    : /content/drive/MyDrive/SPP_GAN_Project/metadata

Notebook 06 Configuration Completed Successfully.


In [90]:
# =============================================================================
# Part 6.3 : Tensor Dataset Preparation
# Block 1 : Validate Dataset Structure
# =============================================================================

print("=" * 80)
print("Validating Datasets")
print("=" * 80)

dataset_metadata = {}

for name, df in datasets.items():

    print(f"\n{name}")
    print("-" * 60)

    print(f"Rows            : {df.shape[0]}")
    print(f"Columns         : {df.shape[1]}")
    print(f"Missing Values  : {df.isnull().sum().sum()}")
    print(f"Duplicate Rows  : {df.duplicated().sum()}")

    dataset_metadata[name] = {
        "rows": df.shape[0],
        "columns": df.shape[1]
    }

print("\nDataset validation completed.")

Validating Datasets

Adult Income
------------------------------------------------------------
Rows            : 32510
Columns         : 15
Missing Values  : 0
Duplicate Rows  : 0

Bank Marketing
------------------------------------------------------------
Rows            : 45211
Columns         : 17
Missing Values  : 0
Duplicate Rows  : 0

Breast Cancer
------------------------------------------------------------
Rows            : 569
Columns         : 32
Missing Values  : 0
Duplicate Rows  : 0

Dataset validation completed.


In [91]:
# =============================================================================
# Block 2 : Feature / Target Separation
# =============================================================================

TARGET_COLUMNS = {

    "Adult Income": "income",

    "Bank Marketing": "y",

    "Breast Cancer": "diagnosis"

}

processed_data = {}

print("=" * 80)
print("Separating Features and Targets")
print("=" * 80)

for name, df in datasets.items():

    target = TARGET_COLUMNS[name]

    if target not in df.columns:
        raise ValueError(
            f"Target column '{target}' not found in {name}"
        )

    X = df.drop(columns=[target])

    y = df[target]

    processed_data[name] = {
        "X": X,
        "y": y
    }

    print(f"{name:<20} X:{X.shape}  y:{y.shape}")

Separating Features and Targets
Adult Income         X:(32510, 14)  y:(32510,)
Bank Marketing       X:(45211, 16)  y:(45211,)
Breast Cancer        X:(569, 31)  y:(569,)


In [92]:
# =============================================================================
# Block 3 : NumPy Conversion
# =============================================================================

numpy_data = {}

print("=" * 80)
print("Converting to NumPy Arrays")
print("=" * 80)

for name, data in processed_data.items():

    X = data["X"].astype(np.float32).to_numpy()

    y = data["y"].astype(np.float32).to_numpy()

    numpy_data[name] = {

        "X": X,

        "y": y

    }

    print(f"{name:<20} {X.shape}")

Converting to NumPy Arrays
Adult Income         (32510, 14)
Bank Marketing       (45211, 16)
Breast Cancer        (569, 31)


In [93]:
# =============================================================================
# Block 4 : Tensor Conversion
# =============================================================================

tensor_data = {}

print("=" * 80)
print("Creating PyTorch Tensors")
print("=" * 80)

for name, data in numpy_data.items():

    X_tensor = torch.tensor(
        data["X"],
        dtype=torch.float32
    )

    y_tensor = torch.tensor(
        data["y"],
        dtype=torch.float32
    )

    tensor_data[name] = {

        "X": X_tensor,

        "y": y_tensor

    }

    print(f"{name:<20} {X_tensor.shape}")

Creating PyTorch Tensors
Adult Income         torch.Size([32510, 14])
Bank Marketing       torch.Size([45211, 16])
Breast Cancer        torch.Size([569, 31])


In [94]:
# =============================================================================
# Block 5 : TensorDataset
# =============================================================================

tensor_datasets = {}

print("=" * 80)
print("Creating TensorDataset Objects")
print("=" * 80)

for name, data in tensor_data.items():

    dataset = TensorDataset(

        data["X"],

        data["y"]

    )

    tensor_datasets[name] = dataset

    print(f"{name:<20} Samples : {len(dataset)}")

Creating TensorDataset Objects
Adult Income         Samples : 32510
Bank Marketing       Samples : 45211
Breast Cancer        Samples : 569


In [95]:
# =============================================================================
# Block 6 : DataLoaders
# =============================================================================

train_loaders = {}

print("=" * 80)
print("Creating DataLoaders")
print("=" * 80)

for name, dataset in tensor_datasets.items():

    loader = DataLoader(

        dataset,

        batch_size=BATCH_SIZE,

        shuffle=True,

        drop_last=True,

        pin_memory=torch.cuda.is_available()

    )

    train_loaders[name] = loader

    print(
        f"{name:<20} "
        f"{len(loader)} batches"
    )

Creating DataLoaders
Adult Income         126 batches
Bank Marketing       176 batches
Breast Cancer        2 batches


In [96]:
# =============================================================================
# Block 7 : Save Metadata
# =============================================================================

tensor_metadata = {}

for name, data in tensor_data.items():

    tensor_metadata[name] = {

        "input_dimension": data["X"].shape[1],

        "num_samples": data["X"].shape[0],

        "batch_size": BATCH_SIZE

    }

metadata_path = METADATA_DIR / "tensor_metadata.pkl"

joblib.dump(

    tensor_metadata,

    metadata_path

)

print("=" * 80)
print("Tensor Metadata Saved Successfully")
print("=" * 80)

print(metadata_path)

Tensor Metadata Saved Successfully
/content/drive/MyDrive/SPP_GAN_Project/metadata/tensor_metadata.pkl


In [97]:
# =============================================================================
# Part 6.4 : Differential Privacy Configuration
# Block 1 : Load Privacy Parameters
# =============================================================================

print("=" * 80)
print("Loading Differential Privacy Parameters")
print("=" * 80)

DP_CONFIG = {

    "epsilon": EPSILON,

    "delta": DELTA,

    "noise_multiplier": NOISE_MULTIPLIER,

    "max_grad_norm": MAX_GRAD_NORM,

    "secure_mode": False

}

for key, value in DP_CONFIG.items():

    print(f"{key:<20}: {value}")

Loading Differential Privacy Parameters
epsilon             : 4
delta               : 1e-05
noise_multiplier    : 1.1
max_grad_norm       : 1.0
secure_mode         : False


In [98]:
# =============================================================================
# Block 2 : Validate Privacy Parameters
# =============================================================================

assert DP_CONFIG["epsilon"] > 0, \
    "Privacy budget (ε) must be positive."

assert DP_CONFIG["delta"] > 0, \
    "Delta must be positive."

assert DP_CONFIG["noise_multiplier"] > 0, \
    "Noise multiplier must be positive."

assert DP_CONFIG["max_grad_norm"] > 0, \
    "Gradient clipping value must be positive."

print("=" * 80)
print("Privacy Parameters Validated Successfully")
print("=" * 80)

Privacy Parameters Validated Successfully


In [99]:
# =============================================================================
# Block 3 : Privacy Summary
# =============================================================================

privacy_summary = pd.DataFrame({

    "Parameter": [

        "Privacy Budget (ε)",

        "Delta (δ)",

        "Noise Multiplier",

        "Gradient Clipping",

        "Secure Mode"

    ],

    "Value": [

        DP_CONFIG["epsilon"],

        DP_CONFIG["delta"],

        DP_CONFIG["noise_multiplier"],

        DP_CONFIG["max_grad_norm"],

        DP_CONFIG["secure_mode"]

    ]

})

display(privacy_summary)

,Parameter,Value
0,Privacy Budget (ε),4
1,Delta (δ),0.00001
2,Noise Multiplier,1.1
3,Gradient Clipping,1.0
4,Secure Mode,False


In [100]:
# =============================================================================
# Block 4 : Save Privacy Configuration
# =============================================================================

privacy_file = METADATA_DIR / "privacy_config.pkl"

joblib.dump(

    DP_CONFIG,

    privacy_file

)

print("=" * 80)
print("Privacy Configuration Saved")
print("=" * 80)

print(privacy_file)

Privacy Configuration Saved
/content/drive/MyDrive/SPP_GAN_Project/metadata/privacy_config.pkl


In [101]:
# =============================================================================
# Block 5 : Privacy Report
# =============================================================================

print("=" * 80)
print("Differential Privacy Report")
print("=" * 80)

print(f"Privacy Budget (ε) : {DP_CONFIG['epsilon']}")

print(f"Delta (δ)          : {DP_CONFIG['delta']}")

print(f"Noise Multiplier   : {DP_CONFIG['noise_multiplier']}")

print(f"Gradient Clip      : {DP_CONFIG['max_grad_norm']}")

print(f"Secure Mode        : {DP_CONFIG['secure_mode']}")

Differential Privacy Report
Privacy Budget (ε) : 4
Delta (δ)          : 1e-05
Noise Multiplier   : 1.1
Gradient Clip      : 1.0
Secure Mode        : False


In [102]:
# =============================================================================
# Block 6 : DP Readiness
# =============================================================================

print("=" * 80)
print("Differential Privacy Readiness")
print("=" * 80)

print("✓ Privacy parameters loaded")

print("✓ Parameters validated")

print("✓ Configuration saved")

print("✓ Ready for Opacus PrivacyEngine")

print("\nNotebook ready for Generator construction.")

Differential Privacy Readiness
✓ Privacy parameters loaded
✓ Parameters validated
✓ Configuration saved
✓ Ready for Opacus PrivacyEngine

Notebook ready for Generator construction.


In [103]:
# =============================================================================
# Part 6.5
# Block 1 : Generator Configuration
# =============================================================================

print("=" * 80)
print("Generator Configuration")
print("=" * 80)

GENERATOR_CONFIG = {

    "latent_dim": LATENT_DIM,

    "hidden_dims": [256, 512, 512, 256],

    "dropout": 0.20,

    "negative_slope": 0.2

}

print(GENERATOR_CONFIG)

Generator Configuration
{'latent_dim': 128, 'hidden_dims': [256, 512, 512, 256], 'dropout': 0.2, 'negative_slope': 0.2}


In [104]:
# =============================================================================
# Block 2 : Xavier Initialization
# =============================================================================

def initialize_weights(module):

    if isinstance(module, nn.Linear):

        nn.init.xavier_uniform_(module.weight)

        if module.bias is not None:

            nn.init.zeros_(module.bias)

In [105]:
# =============================================================================
# Block 3 : Generator
# =============================================================================

class Generator(nn.Module):

    def __init__(
        self,
        latent_dim,
        output_dim,
        hidden_dims,
        dropout=0.2
    ):

        super().__init__()

        layers = []

        in_features = latent_dim

        for hidden in hidden_dims:

            layers.extend([

                nn.Linear(in_features, hidden),

                nn.LayerNorm(hidden),

                nn.LeakyReLU(
                    negative_slope=0.2,
                    inplace=True
                ),

                nn.Dropout(dropout)

            ])

            in_features = hidden

        layers.append(

            nn.Linear(
                in_features,
                output_dim
            )

        )

        layers.append(

            nn.Tanh()

        )

        self.model = nn.Sequential(*layers)

        self.apply(initialize_weights)

    def forward(self, z):

        return self.model(z)

In [106]:
# =============================================================================
# Block 4 : Initialize Generators
# =============================================================================

generators = {}

for dataset_name, meta in tensor_metadata.items():

    generators[dataset_name] = Generator(

        latent_dim=LATENT_DIM,

        output_dim=meta["input_dimension"],

        hidden_dims=GENERATOR_CONFIG["hidden_dims"],

        dropout=GENERATOR_CONFIG["dropout"]

    ).to(DEVICE)

print("=" * 80)
print("Generators Created Successfully")
print("=" * 80)

for name in generators:

    print(name)

Generators Created Successfully
Adult Income
Bank Marketing
Breast Cancer


In [107]:
# =============================================================================
# Block 5 : Model Summary
# =============================================================================

print("=" * 80)
print("Adult Income Generator")
print("=" * 80)

print(

    generators["Adult Income"]

)

Adult Income Generator
Generator(
  (model): Sequential(
    (0): Linear(in_features=128, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (2): LeakyReLU(negative_slope=0.2, inplace=True)
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=256, out_features=512, bias=True)
    (5): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (6): LeakyReLU(negative_slope=0.2, inplace=True)
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=512, out_features=512, bias=True)
    (9): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (10): LeakyReLU(negative_slope=0.2, inplace=True)
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=512, out_features=256, bias=True)
    (13): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (14): LeakyReLU(negative_slope=0.2, inplace=True)
    (15): Dropout(p=0.2, inplace=False)
    (16): Linear(in_features=256, out_features=14, bias=True)
    (17

In [108]:
# =============================================================================
# Block 6 : Forward Pass
# =============================================================================

dataset = "Adult Income"

generator = generators[dataset]

generator.train()

noise = torch.randn(

    16,

    LATENT_DIM,

    device=DEVICE

)

with torch.no_grad():

    fake = generator(noise)

print("=" * 80)
print("Generator Verification")
print("=" * 80)

print("Noise Shape :", noise.shape)

print("Output Shape:", fake.shape)

assert fake.shape == (

    16,

    tensor_metadata[dataset]["input_dimension"]

)

print("\nGenerator Ready")

Generator Verification
Noise Shape : torch.Size([16, 128])
Output Shape: torch.Size([16, 14])

Generator Ready


In [109]:
# =============================================================================
# Part 6.6
# Block 1 : Discriminator Configuration
# =============================================================================

print("=" * 80)
print("Discriminator Configuration")
print("=" * 80)

DISCRIMINATOR_CONFIG = {

    "hidden_dims": [512, 256, 128],

    "dropout": 0.30,

    "negative_slope": 0.2

}

print(DISCRIMINATOR_CONFIG)

Discriminator Configuration
{'hidden_dims': [512, 256, 128], 'dropout': 0.3, 'negative_slope': 0.2}


In [110]:
# =============================================================================
# Block 2 : Weight Initialization
# =============================================================================

def initialize_discriminator_weights(module):

    if isinstance(module, nn.Linear):

        nn.init.xavier_uniform_(module.weight)

        if module.bias is not None:

            nn.init.zeros_(module.bias)

In [111]:
# =============================================================================
# Block 3 : Discriminator Network
# =============================================================================

class Discriminator(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dims,
        dropout=0.30
    ):

        super().__init__()

        layers = []

        in_features = input_dim

        for hidden in hidden_dims:

            layers.extend([

                nn.Linear(in_features, hidden),

                nn.LayerNorm(hidden),

                nn.LeakyReLU(
                    negative_slope=0.2,
                    inplace=True
                ),

                nn.Dropout(dropout)

            ])

            in_features = hidden

        # Output raw logits
        layers.append(

            nn.Linear(in_features, 1)

        )

        self.model = nn.Sequential(*layers)

        self.apply(initialize_discriminator_weights)

    def forward(self, x):

        return self.model(x)

In [112]:
# =============================================================================
# Block 4 : Initialize Discriminators
# =============================================================================

discriminators = {}

for dataset_name, meta in tensor_metadata.items():

    discriminators[dataset_name] = Discriminator(

        input_dim=meta["input_dimension"],

        hidden_dims=DISCRIMINATOR_CONFIG["hidden_dims"],

        dropout=DISCRIMINATOR_CONFIG["dropout"]

    ).to(DEVICE)

print("=" * 80)
print("Discriminators Created Successfully")
print("=" * 80)

for name in discriminators:

    print(name)

Discriminators Created Successfully
Adult Income
Bank Marketing
Breast Cancer


In [113]:
# =============================================================================
# Block 5 : Model Summary
# =============================================================================

print("=" * 80)
print("Adult Income Discriminator")
print("=" * 80)

print(

    discriminators["Adult Income"]

)

Adult Income Discriminator
Discriminator(
  (model): Sequential(
    (0): Linear(in_features=14, out_features=512, bias=True)
    (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (2): LeakyReLU(negative_slope=0.2, inplace=True)
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (6): LeakyReLU(negative_slope=0.2, inplace=True)
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=256, out_features=128, bias=True)
    (9): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (10): LeakyReLU(negative_slope=0.2, inplace=True)
    (11): Dropout(p=0.3, inplace=False)
    (12): Linear(in_features=128, out_features=1, bias=True)
  )
)


In [114]:
# =============================================================================
# Block 6 : Verification
# =============================================================================

dataset = "Adult Income"

discriminator = discriminators[dataset]

discriminator.train()

sample = torch.randn(

    16,

    tensor_metadata[dataset]["input_dimension"],

    device=DEVICE

)

with torch.no_grad():

    output = discriminator(sample)

print("=" * 80)
print("Discriminator Verification")
print("=" * 80)

print("Input Shape :", sample.shape)

print("Output Shape:", output.shape)

assert output.shape == (16, 1)

print("\nDiscriminator Ready")

Discriminator Verification
Input Shape : torch.Size([16, 14])
Output Shape: torch.Size([16, 1])

Discriminator Ready


In [115]:
# =============================================================================
# Part 6.7
# Block 1 : Loss Function
# =============================================================================

print("=" * 80)
print("Initializing Loss Function")
print("=" * 80)

criterion = nn.BCEWithLogitsLoss()

print("Loss Function")

print(criterion)

Initializing Loss Function
Loss Function
BCEWithLogitsLoss()


In [116]:
# =============================================================================
# Block 2 : Generator Optimizers
# =============================================================================

generator_optimizers = {}

for dataset_name, generator in generators.items():

    generator_optimizers[dataset_name] = torch.optim.Adam(

        generator.parameters(),

        lr=LEARNING_RATE,

        betas=(BETA1, BETA2)

    )

print("=" * 80)
print("Generator Optimizers Created")
print("=" * 80)

print(generator_optimizers.keys())

Generator Optimizers Created
dict_keys(['Adult Income', 'Bank Marketing', 'Breast Cancer'])


In [117]:
# =============================================================================
# Block 3 : Discriminator Optimizers
# =============================================================================

discriminator_optimizers = {}

for dataset_name, discriminator in discriminators.items():

    discriminator_optimizers[dataset_name] = torch.optim.Adam(

        discriminator.parameters(),

        lr=LEARNING_RATE,

        betas=(BETA1, BETA2)

    )

print("=" * 80)
print("Discriminator Optimizers Created")
print("=" * 80)

print(discriminator_optimizers.keys())

Discriminator Optimizers Created
dict_keys(['Adult Income', 'Bank Marketing', 'Breast Cancer'])


In [118]:
# =============================================================================
# Block 4 : Learning Rate Scheduler
# =============================================================================

generator_schedulers = {}

discriminator_schedulers = {}

for dataset_name in generators.keys():

    generator_schedulers[dataset_name] = torch.optim.lr_scheduler.StepLR(

        generator_optimizers[dataset_name],

        step_size=100,

        gamma=0.5

    )

    discriminator_schedulers[dataset_name] = torch.optim.lr_scheduler.StepLR(

        discriminator_optimizers[dataset_name],

        step_size=100,

        gamma=0.5

    )

print("=" * 80)
print("Learning Rate Schedulers Created")
print("=" * 80)

Learning Rate Schedulers Created


In [119]:
# =============================================================================
# Block 5 : Mixed Precision
# =============================================================================

USE_AMP = DEVICE.type == "cuda"

if USE_AMP:

    from torch.cuda.amp import GradScaler

    scaler = GradScaler()

    print("Automatic Mixed Precision Enabled")

else:

    scaler = None

    print("Running on CPU (AMP Disabled)")

Running on CPU (AMP Disabled)


In [120]:
# =============================================================================
# Block 6 : Training State
# =============================================================================

training_state = {}

for dataset_name in generators.keys():

    training_state[dataset_name] = {

        "generator": generators[dataset_name],

        "discriminator": discriminators[dataset_name],

        "g_optimizer": generator_optimizers[dataset_name],

        "d_optimizer": discriminator_optimizers[dataset_name],

        "g_scheduler": generator_schedulers[dataset_name],

        "d_scheduler": discriminator_schedulers[dataset_name],

        "criterion": criterion,

        "train_loader": train_loaders[dataset_name], # Added the train_loader here

        "generator_loss": [],

        "discriminator_loss": [],

        "epsilon": [],

        "best_generator_loss": float("inf"),

        "best_discriminator_loss": float("inf"),

        "epoch": 0

    }

print("=" * 80)
print("Training State Created")
print("=" * 80)

print(training_state.keys())

Training State Created
dict_keys(['Adult Income', 'Bank Marketing', 'Breast Cancer'])


In [121]:
# =============================================================================
# Block 7 : Verification
# =============================================================================

print("=" * 80)
print("Training Components Summary")
print("=" * 80)

print(f"Datasets              : {len(training_state)}")

print(f"Generator Models      : {len(generators)}")

print(f"Discriminator Models  : {len(discriminators)}")

print(f"Loss Function         : {criterion.__class__.__name__}")

print(f"Optimizer             : Adam")

print(f"Scheduler             : StepLR")

print(f"Mixed Precision       : {USE_AMP}")

print(f"Learning Rate         : {LEARNING_RATE}")

print(f"Epochs                : {EPOCHS}")

print(f"Batch Size            : {BATCH_SIZE}")

print("\nTraining Components Ready.")

Training Components Summary
Datasets              : 3
Generator Models      : 3
Discriminator Models  : 3
Loss Function         : BCEWithLogitsLoss
Optimizer             : Adam
Scheduler             : StepLR
Mixed Precision       : False
Learning Rate         : 0.0002
Epochs                : 300
Batch Size            : 256

Training Components Ready.


In [122]:
# ============================================================
# 6.8.1 Training Components Imports
# ============================================================

import torch
import torch.nn as nn

import os
import json
import time


print("Training component libraries loaded")

Training component libraries loaded


In [123]:
# ============================================================
# 6.8.2 Loss Function
# ============================================================


criterion = nn.BCELoss()


print(
    "Loss Function:",
    criterion
)

Loss Function: BCELoss()


In [124]:
print("============================================================")
print("Optimizer Verification")
print("============================================================")

for dataset_name, state in training_state.items():
    print(f"\nDataset: {dataset_name}")
    print(f"  Generator Optimizer: {state['g_optimizer'].__class__.__name__}")
    print(f"  Discriminator Optimizer: {state['d_optimizer'].__class__.__name__}")

Optimizer Verification

Dataset: Adult Income
  Generator Optimizer: Adam
  Discriminator Optimizer: Adam

Dataset: Bank Marketing
  Generator Optimizer: Adam
  Discriminator Optimizer: Adam

Dataset: Breast Cancer
  Generator Optimizer: Adam
  Discriminator Optimizer: Adam


In [125]:
# ============================================================
# 6.8.4 Device Configuration
# ============================================================

# Update config['device'] to match the actual detected device (from uvJccBcnv6Uf)
config["device"] = DEVICE.type

# Move models to the correct device.
# Models are stored within the training_state dictionary for each dataset.
for dataset_name, state in training_state.items():
    state["generator"] = state["generator"].to(DEVICE)
    state["discriminator"] = state["discriminator"].to(DEVICE)

print(
    "Training Device:",
    DEVICE
)

Training Device: cpu


In [126]:
# ============================================================
# 6.8.5 AMP Configuration
# ============================================================


use_amp = True if DEVICE.type == "cuda" else False


scaler_amp = torch.cuda.amp.GradScaler(
    enabled=use_amp
)


print(
    "AMP Enabled:",
    use_amp
)

AMP Enabled: False


In [127]:
# ============================================================
# 6.8.6 Training History Initialization
# ============================================================


training_history = {


    "epoch": [],


    "generator_loss": [],


    "discriminator_loss": [],


    "privacy_epsilon": [],


    "training_time": []

}



print(
    "Training history initialized"
)

Training history initialized


In [131]:
# ============================================================
# 6.10.3 Attach PrivacyEngine
# ============================================================

for dataset_name, state in training_state.items():

    privacy_engine = PrivacyEngine()

    (
        state["generator"],
        state["g_optimizer"],
        state["train_loader"],
    ) = privacy_engine.make_private(
        module=state["generator"],
        optimizer=state["g_optimizer"],
        data_loader=state["train_loader"],
        noise_multiplier=NOISE_MULTIPLIER,
        max_grad_norm=MAX_GRAD_NORM,
    )

    state["privacy_engine"] = privacy_engine

    print(f"{dataset_name}: PrivacyEngine attached successfully.")

Adult Income: PrivacyEngine attached successfully.
Bank Marketing: PrivacyEngine attached successfully.
Breast Cancer: PrivacyEngine attached successfully.


In [132]:
# ============================================================
# 6.10.4 Verify PrivacyEngine
# ============================================================

for dataset_name, state in training_state.items():

    print(f"\nDataset : {dataset_name}")

    print(
        "Privacy Engine:",
        isinstance(state["privacy_engine"], PrivacyEngine)
    )


Dataset : Adult Income
Privacy Engine: True

Dataset : Bank Marketing
Privacy Engine: True

Dataset : Breast Cancer
Privacy Engine: True


In [133]:
# ============================================================
# 6.8.8 Checkpoint Directory
# ============================================================


CHECKPOINT_DIR = os.path.join(

    PROJECT_DIR,

    "models",

    "dp_gan",

    "checkpoints"

)


os.makedirs(

    CHECKPOINT_DIR,

    exist_ok=True

)


print(
    "Checkpoint directory:"
)


print(
    CHECKPOINT_DIR
)

Checkpoint directory:
/content/drive/MyDrive/SPP_GAN_Project/models/dp_gan/checkpoints


In [134]:
# ============================================================
# 6.8.9 Training Configuration Summary
# ============================================================

# Define DATA_DIM using the input dimension of the 'Adult Income' dataset
# This is used as a representative value for the general training configuration summary.
DATA_DIM = tensor_metadata["Adult Income"]["input_dimension"]

training_config = {


    "epochs":
    config["epochs"],


    "batch_size":
    config["batch_size"],


    "learning_rate":
    config["learning_rate"],


    "optimizer":
    config["optimizer"],


    "latent_dimension":
    LATENT_DIM,


    "data_dimension":
    DATA_DIM


}



print(
    json.dumps(
        training_config,
        indent=4
    )
)

{
    "epochs": 300,
    "batch_size": 256,
    "learning_rate": 0.0002,
    "optimizer": "Adam",
    "latent_dimension": 128,
    "data_dimension": 14
}


In [135]:
# ============================================================
# 6.8.10 Training Components Verification
# ============================================================


print(
    "Generator Parameters:",
    sum(
        p.numel()
        for p in generator.parameters()
    )
)


print(
    "Discriminator Parameters:",
    sum(
        p.numel()
        for p in discriminator.parameters()
    )
)


print(
    "\nTraining Components Ready ✓"
)

Generator Parameters: 569631
Discriminator Parameters: 182529

Training Components Ready ✓


In [136]:
# ============================================================
# 6.9.1 Model Summary
# ============================================================

print("=" * 70)
print("GENERATOR")
print("=" * 70)
print(generator)

print("\n")

print("=" * 70)
print("DISCRIMINATOR")
print("=" * 70)
print(discriminator)

GENERATOR
Generator(
  (model): Sequential(
    (0): Linear(in_features=128, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (2): LeakyReLU(negative_slope=0.2, inplace=True)
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=256, out_features=512, bias=True)
    (5): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (6): LeakyReLU(negative_slope=0.2, inplace=True)
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=512, out_features=512, bias=True)
    (9): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (10): LeakyReLU(negative_slope=0.2, inplace=True)
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=512, out_features=256, bias=True)
    (13): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (14): LeakyReLU(negative_slope=0.2, inplace=True)
    (15): Dropout(p=0.2, inplace=False)
    (16): Linear(in_features=256, out_features=31, bias=True)
    (17): Tanh()
  )

In [137]:
# ============================================================
# 6.9.2 Trainable Parameters
# ============================================================

generator_params = sum(
    p.numel() for p in generator.parameters()
    if p.requires_grad
)

discriminator_params = sum(
    p.numel() for p in discriminator.parameters()
    if p.requires_grad
)

print(f"Generator Parameters     : {generator_params:,}")
print(f"Discriminator Parameters : {discriminator_params:,}")

Generator Parameters     : 569,631
Discriminator Parameters : 182,529


In [138]:
# ============================================================
# 6.9.3 Generator Forward Pass
# ============================================================

generator.eval()

with torch.no_grad():

    noise = torch.randn(
        8,
        LATENT_DIM,
        device=DEVICE
    )

    fake_samples = generator(noise)

print("Noise Shape       :", noise.shape)
print("Generated Shape   :", fake_samples.shape)

Noise Shape       : torch.Size([8, 128])
Generated Shape   : torch.Size([8, 31])


In [139]:
# ============================================================
# 6.9.4 Discriminator Forward Pass
# ============================================================

discriminator.eval()

with torch.no_grad():

    predictions = discriminator(fake_samples)

print("Discriminator Output Shape :", predictions.shape)

Discriminator Output Shape : torch.Size([8, 1])


In [140]:
# ============================================================
# 6.9.5 Numerical Stability Check
# ============================================================

assert not torch.isnan(fake_samples).any(), \
    "Generator produced NaN values."

assert not torch.isinf(fake_samples).any(), \
    "Generator produced Inf values."

assert not torch.isnan(predictions).any(), \
    "Discriminator produced NaN values."

assert not torch.isinf(predictions).any(), \
    "Discriminator produced Inf values."

print("✓ Numerical stability check passed.")

✓ Numerical stability check passed.


In [141]:
# ============================================================
# 6.9.6 Training Batch Validation
# ============================================================

# Using the DataLoader for 'Adult Income' dataset for validation
batch = next(iter(train_loaders['Adult Income']))

print("Batch Shape :", batch[0].shape)
print("Batch Type  :", batch[0].dtype)

assert batch[0].shape[1] == DATA_DIM, \
    "Input dimension mismatch."

print("✓ Batch validation passed.")

Batch Shape : torch.Size([256, 14])
Batch Type  : torch.float32
✓ Batch validation passed.


In [142]:
# ============================================================
# 6.9.7 Device Validation
# ============================================================

print("Current Device :", DEVICE)

print("Generator Device :",
      next(generator.parameters()).device)

print("Discriminator Device :",
      next(discriminator.parameters()).device)

Current Device : cpu
Generator Device : cpu
Discriminator Device : cpu


In [143]:
# ============================================================
# 6.9.8 Validation Summary
# ============================================================

print("=" * 70)
print("MODEL VALIDATION COMPLETED SUCCESSFULLY")
print("=" * 70)

print(f"Data Dimension      : {DATA_DIM}")
print(f"Latent Dimension    : {LATENT_DIM}")
print(f"Training Batches    : {len(train_loaders['Adult Income'])}")
print(f"Device              : {DEVICE}")

print("=" * 70)

MODEL VALIDATION COMPLETED SUCCESSFULLY
Data Dimension      : 14
Latent Dimension    : 128
Training Batches    : 126
Device              : cpu


In [144]:
# ============================================================
# 6.10.1 Import PrivacyEngine
# ============================================================

from opacus import PrivacyEngine

print("Opacus imported successfully.")

Opacus imported successfully.


In [145]:
# ============================================================
# 6.10.2 Privacy Configuration
# ============================================================

EPSILON = config["privacy_budget"]
DELTA = config["delta"]
NOISE_MULTIPLIER = config["noise_multiplier"]
MAX_GRAD_NORM = config["gradient_clip"]

print(f"Privacy Budget (ε): {EPSILON}")
print(f"Delta (δ): {DELTA}")
print(f"Noise Multiplier : {NOISE_MULTIPLIER}")
print(f"Max Grad Norm    : {MAX_GRAD_NORM}")

Privacy Budget (ε): 4
Delta (δ): 1e-05
Noise Multiplier : 1.1
Max Grad Norm    : 1.0


In [147]:
# ============================================================
# 6.10.3 Attach PrivacyEngine
# ============================================================

for dataset_name, state in training_state.items():

    privacy_engine = PrivacyEngine()

    (
        state["generator"],
        state["g_optimizer"],
        state["train_loader"],
    ) = privacy_engine.make_private(
        module=state["generator"],
        optimizer=state["g_optimizer"],
        data_loader=state["train_loader"],
        noise_multiplier=NOISE_MULTIPLIER,
        max_grad_norm=MAX_GRAD_NORM,
    )

    state["privacy_engine"] = privacy_engine

    print(f"{dataset_name}: PrivacyEngine attached successfully.")

Adult Income: PrivacyEngine attached successfully.
Bank Marketing: PrivacyEngine attached successfully.
Breast Cancer: PrivacyEngine attached successfully.


In [148]:
# ============================================================
# 6.10.4 Verify PrivacyEngine
# ============================================================

for dataset_name, state in training_state.items():

    print(f"\nDataset : {dataset_name}")

    print(
        "Privacy Engine:",
        isinstance(state["privacy_engine"], PrivacyEngine)
    )


Dataset : Adult Income
Privacy Engine: True

Dataset : Bank Marketing
Privacy Engine: True

Dataset : Breast Cancer
Privacy Engine: True


In [149]:
# ============================================================
# 6.10.5 Summary
# ============================================================

print("=" * 60)
print("PrivacyEngine successfully attached.")
print("=" * 60)

print(f"Noise Multiplier : {NOISE_MULTIPLIER}")
print(f"Gradient Clip    : {MAX_GRAD_NORM}")
print(f"Target Delta     : {DELTA}")

PrivacyEngine successfully attached.
Noise Multiplier : 1.1
Gradient Clip    : 1.0
Target Delta     : 1e-05


In [169]:
# ============================================================
# 6.12 Batch Training
# ============================================================

def train_batch(
    generator,
    discriminator,
    batch,
    optimizer_G,
    optimizer_D,
    criterion,
    latent_dim,
    device,
    privacy_engine=None,
    scaler=None
):
    """
    Train one mini-batch of DP-GAN.

    Returns
    -------
    dict
        Training statistics
    """

    generator.train()
    discriminator.train()

    # --------------------------------------------------------
    # Load real data
    # --------------------------------------------------------
    if isinstance(batch, (list, tuple)):
        real_data = batch[0]
    else:
        real_data = batch

    real_data = real_data.float().to(device)

    batch_size = real_data.size(0)

    real_labels = torch.ones(batch_size, 1, device=device)
    fake_labels = torch.zeros(batch_size, 1, device=device)

    ###########################################################
    # Train Discriminator
    ###########################################################

    optimizer_D.zero_grad()

    noise = torch.randn(
        batch_size,
        latent_dim,
        device=device
    )

    if scaler is not None:

        with torch.cuda.amp.autocast():

            fake_data = generator(noise)

            real_output = discriminator(real_data)
            fake_output = discriminator(fake_data.detach())

            d_real_loss = criterion(
                real_output,
                real_labels
            )

            d_fake_loss = criterion(
                fake_output,
                fake_labels
            )

            d_loss = (
                d_real_loss +
                d_fake_loss
            ) / 2

        scaler.scale(d_loss).backward()
        scaler.step(optimizer_D)
        scaler.update()

    else:

        fake_data = generator(noise)

        real_output = discriminator(real_data)

        fake_output = discriminator(
            fake_data.detach()
        )

        d_real_loss = criterion(
            real_output,
            real_labels
        )

        d_fake_loss = criterion(
            fake_output,
            fake_labels
        )

        d_loss = (
            d_real_loss +
            d_fake_loss
        ) / 2

        d_loss.backward()

        optimizer_D.step()

    ###########################################################
    # Train Generator
    ###########################################################

    optimizer_G.zero_grad()

    noise = torch.randn(
        batch_size,
        latent_dim,
        device=device
    )

    if scaler is not None:

        with torch.cuda.amp.autocast():

            generated = generator(noise)

            predictions = discriminator(
                generated
            )

            g_loss = criterion(
                predictions,
                real_labels
            )

        scaler.scale(g_loss).backward()

        scaler.step(optimizer_G)

        scaler.update()

    else:

        generated = generator(noise)

        predictions = discriminator(
            generated
        )

        g_loss = criterion(
            predictions,
            real_labels
        )

        g_loss.backward()

        optimizer_G.step()

    ###########################################################
    # Privacy Budget
    ###########################################################

    epsilon = None

    if privacy_engine is not None:

        try:

            epsilon = privacy_engine.get_epsilon(
                delta=1e-5
            )

        except Exception:

            epsilon = None

    ###########################################################
    # Accuracy (optional)
    ###########################################################

    with torch.no_grad():

        real_acc = (
            (torch.sigmoid(real_output) > 0.5)
            .float()
            .eq(real_labels)
            .float()
            .mean()
            .item()
        )

        fake_acc = (
            (torch.sigmoid(fake_output) > 0.5)
            .float()
            .eq(fake_labels)
            .float()
            .mean()
            .item()
        )

    ###########################################################
    # Return statistics
    ###########################################################

    return {

        "generator_loss": float(g_loss.item()),

        "discriminator_loss": float(d_loss.item()),

        "real_loss": float(d_real_loss.item()),

        "fake_loss": float(d_fake_loss.item()),

        "real_accuracy": real_acc,

        "fake_accuracy": fake_acc,

        "epsilon": epsilon,

        "batch_size": batch_size

    }

In [173]:
# ============================================================
# 6.8.6 Epoch Training
# ============================================================

def train_epoch(
    generator,
    discriminator,
    train_loader,
    optimizer_G,
    optimizer_D,
    criterion,
    latent_dim,
    device,
    privacy_engine=None,
    scaler=None,
    scheduler_G=None,
    scheduler_D=None,
    epoch=1,
    total_epochs=1
):
    """
    Train one complete epoch.

    Returns
    -------
    dict
        Epoch statistics.
    """

    generator.train()
    discriminator.train()

    # ========================================================
    # Initialize Statistics
    # ========================================================

    total_g_loss = 0.0
    total_d_loss = 0.0
    total_real_loss = 0.0
    total_fake_loss = 0.0

    total_real_acc = 0.0
    total_fake_acc = 0.0

    num_batches = 0

    # ========================================================
    # Loop over batches
    # ========================================================

    for batch in train_loader:

        stats = train_batch(
            generator=generator,
            discriminator=discriminator,
            batch=batch,
            optimizer_G=optimizer_G,
            optimizer_D=optimizer_D,
            criterion=criterion,
            latent_dim=latent_dim,
            device=device,
            privacy_engine=privacy_engine,
            scaler=scaler
        )

        total_g_loss += stats["generator_loss"]
        total_d_loss += stats["discriminator_loss"]

        total_real_loss += stats["real_loss"]
        total_fake_loss += stats["fake_loss"]

        total_real_acc += stats["real_accuracy"]
        total_fake_acc += stats["fake_accuracy"]

        num_batches += 1

    # ========================================================
    # Learning Rate Scheduler
    # ========================================================

    if scheduler_G is not None:
        scheduler_G.step()

    if scheduler_D is not None:
        scheduler_D.step()

    # ========================================================
    # Average Statistics
    # ========================================================

    avg_g_loss = total_g_loss / num_batches
    avg_d_loss = total_d_loss / num_batches

    avg_real_loss = total_real_loss / num_batches
    avg_fake_loss = total_fake_loss / num_batches

    avg_real_acc = total_real_acc / num_batches
    avg_fake_acc = total_fake_acc / num_batches

    # ========================================================
    # Privacy Budget
    # ========================================================

    epsilon = None

    if privacy_engine is not None:

        try:

            epsilon = privacy_engine.get_epsilon(
                delta=1e-5
            )

        except Exception:

            epsilon = None

    # ========================================================
    # Print Progress
    # ========================================================

    print("=" * 70)

    print(f"Epoch {epoch}/{total_epochs}")

    print(f"Generator Loss     : {avg_g_loss:.4f}")
    print(f"Discriminator Loss : {avg_d_loss:.4f}")

    print(f"Real Accuracy      : {avg_real_acc:.4f}")
    print(f"Fake Accuracy      : {avg_fake_acc:.4f}")

    if epsilon is not None:
        print(f"Privacy ε          : {epsilon:.4f}")

    print("=" * 70)

    # ========================================================
    # Return Statistics
    # ========================================================

    return {

        "epoch": epoch,

        "generator_loss": avg_g_loss,

        "discriminator_loss": avg_d_loss,

        "real_loss": avg_real_loss,

        "fake_loss": avg_fake_loss,

        "real_accuracy": avg_real_acc,

        "fake_accuracy": avg_fake_acc,

        "epsilon": epsilon

    }

In [174]:
# ============================================================
# 6.8.7 Full Training
# ============================================================

def train_model(
    generator,
    discriminator,
    train_loader,
    optimizer_G,
    optimizer_D,
    criterion,
    latent_dim,
    epochs,
    device,
    privacy_engine=None,
    scaler=None,
    scheduler_G=None,
    scheduler_D=None,
    checkpoint_dir=None
):
    """
    Complete DP-GAN Training Pipeline

    Returns
    -------
    history : dict
        Training history
    """

    history = {

        "epoch": [],
        "generator_loss": [],
        "discriminator_loss": [],
        "real_loss": [],
        "fake_loss": [],
        "real_accuracy": [],
        "fake_accuracy": [],
        "epsilon": []

    }

    best_generator_loss = float("inf")

    print("=" * 80)
    print("Starting DP-GAN Training")
    print("=" * 80)

    for epoch in range(epochs):

        stats = train_epoch(

            generator=generator,

            discriminator=discriminator,

            train_loader=train_loader,

            optimizer_G=optimizer_G,

            optimizer_D=optimizer_D,

            criterion=criterion,

            latent_dim=latent_dim,

            device=device,

            privacy_engine=privacy_engine,

            scaler=scaler,

            scheduler_G=scheduler_G,

            scheduler_D=scheduler_D,

            epoch=epoch + 1,

            total_epochs=epochs

        )

        # --------------------------------------------------
        # Store History
        # --------------------------------------------------

        history["epoch"].append(stats["epoch"])

        history["generator_loss"].append(
            stats["generator_loss"]
        )

        history["discriminator_loss"].append(
            stats["discriminator_loss"]
        )

        history["real_loss"].append(
            stats["real_loss"]
        )

        history["fake_loss"].append(
            stats["fake_loss"]
        )

        history["real_accuracy"].append(
            stats["real_accuracy"]
        )

        history["fake_accuracy"].append(
            stats["fake_accuracy"]
        )

        history["epsilon"].append(
            stats["epsilon"]
        )

        # --------------------------------------------------
        # Save Best Model
        # --------------------------------------------------

        if checkpoint_dir is not None:

            os.makedirs(
                checkpoint_dir,
                exist_ok=True
            )

            if stats["generator_loss"] < best_generator_loss:

                best_generator_loss = stats["generator_loss"]

                torch.save(

                    generator.state_dict(),

                    os.path.join(
                        checkpoint_dir,
                        "best_generator.pt"
                    )

                )

                torch.save(

                    discriminator.state_dict(),

                    os.path.join(
                        checkpoint_dir,
                        "best_discriminator.pt"
                    )

                )

        # --------------------------------------------------
        # Epoch Summary
        # --------------------------------------------------

        print()

        print("-" * 60)

        print(
            f"Epoch {epoch+1}/{epochs} Completed"
        )

        print(
            f"G Loss : {stats['generator_loss']:.4f}"
        )

        print(
            f"D Loss : {stats['discriminator_loss']:.4f}"
        )

        print(
            f"Real Accuracy : {stats['real_accuracy']:.4f}"
        )

        print(
            f"Fake Accuracy : {stats['fake_accuracy']:.4f}"
        )

        if stats["epsilon"] is not None:

            print(
                f"Privacy ε : {stats['epsilon']:.4f}"
            )

        print("-" * 60)

    print()

    print("=" * 80)
    print("DP-GAN Training Finished")
    print("=" * 80)

    return history

In [175]:
# ============================================================
# 6.8.8 Save Trained Models
# ============================================================

import os
import json
import torch
import pandas as pd
from datetime import datetime


def save_trained_models(
    generator,
    discriminator,
    optimizer_G,
    optimizer_D,
    scheduler_G,
    scheduler_D,
    history,
    config,
    save_dir
):
    """
    Save all trained DP-GAN artifacts.

    Parameters
    ----------
    generator : Generator model

    discriminator : Discriminator model

    optimizer_G : Generator optimizer

    optimizer_D : Discriminator optimizer

    scheduler_G : Generator LR scheduler

    scheduler_D : Discriminator LR scheduler

    history : dict

    config : dict

    save_dir : str
    """

    os.makedirs(save_dir, exist_ok=True)

    print("=" * 70)
    print("Saving Trained Models")
    print("=" * 70)

    # --------------------------------------------------------
    # Generator
    # --------------------------------------------------------

    generator_path = os.path.join(
        save_dir,
        "generator.pt"
    )

    torch.save(
        generator.state_dict(),
        generator_path
    )

    print("✓ Generator saved")

    # --------------------------------------------------------
    # Discriminator
    # --------------------------------------------------------

    discriminator_path = os.path.join(
        save_dir,
        "discriminator.pt"
    )

    torch.save(
        discriminator.state_dict(),
        discriminator_path
    )

    print("✓ Discriminator saved")

    # --------------------------------------------------------
    # Optimizers
    # --------------------------------------------------------

    optimizer_path = os.path.join(
        save_dir,
        "optimizers.pt"
    )

    torch.save(
        {
            "generator_optimizer":
                optimizer_G.state_dict(),

            "discriminator_optimizer":
                optimizer_D.state_dict()
        },
        optimizer_path
    )

    print("✓ Optimizers saved")

    # --------------------------------------------------------
    # LR Schedulers
    # --------------------------------------------------------

    scheduler_path = os.path.join(
        save_dir,
        "schedulers.pt"
    )

    torch.save(
        {
            "generator_scheduler":
                scheduler_G.state_dict()
                if scheduler_G is not None else None,

            "discriminator_scheduler":
                scheduler_D.state_dict()
                if scheduler_D is not None else None
        },
        scheduler_path
    )

    print("✓ Schedulers saved")

    # --------------------------------------------------------
    # Training History
    # --------------------------------------------------------

    history_path = os.path.join(
        save_dir,
        "training_history.csv"
    )

    pd.DataFrame(history).to_csv(
        history_path,
        index=False
    )

    print("✓ Training history saved")

    # --------------------------------------------------------
    # Configuration
    # --------------------------------------------------------

    config_path = os.path.join(
        save_dir,
        "config.json"
    )

    with open(config_path, "w") as f:

        json.dump(
            config,
            f,
            indent=4
        )

    print("✓ Configuration saved")

    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    metadata = {

        "created":

            datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            ),

        "epochs":

            len(history["epoch"]),

        "final_generator_loss":

            history["generator_loss"][-1],

        "final_discriminator_loss":

            history["discriminator_loss"][-1],

        "best_generator_loss":

            min(history["generator_loss"]),

        "best_discriminator_loss":

            min(history["discriminator_loss"])

    }

    metadata_path = os.path.join(
        save_dir,
        "metadata.json"
    )

    with open(metadata_path, "w") as f:

        json.dump(
            metadata,
            f,
            indent=4
        )

    print("✓ Metadata saved")

    print("=" * 70)
    print("All training artifacts saved successfully.")
    print("=" * 70)

    return {

        "generator": generator_path,

        "discriminator": discriminator_path,

        "optimizers": optimizer_path,

        "schedulers": scheduler_path,

        "history": history_path,

        "config": config_path,

        "metadata": metadata_path

    }

In [176]:
# ============================================================
# 6.8.9 Generate Synthetic Data
# ============================================================

def generate_synthetic_data(
    generator,
    dataset_name,
    original_dataframe,
    latent_dim,
    device,
    num_samples=None
):
    """
    Generate synthetic tabular data using a trained Generator.

    Parameters
    ----------
    generator : torch.nn.Module
        Trained generator.

    dataset_name : str
        Dataset name.

    original_dataframe : pandas.DataFrame
        Original processed dataset.

    latent_dim : int
        Latent vector dimension.

    device : torch.device

    num_samples : int, optional
        Number of synthetic records.
        Default = original dataset size.

    Returns
    -------
    pandas.DataFrame
    """

    generator.eval()

    if num_samples is None:
        num_samples = len(original_dataframe)

    feature_names = original_dataframe.columns.tolist()

    print("=" * 70)
    print(f"Generating Synthetic Data : {dataset_name}")
    print("=" * 70)

    with torch.no_grad():

        noise = torch.randn(
            num_samples,
            latent_dim,
            device=device
        )

        synthetic = generator(noise)

        synthetic = synthetic.cpu().numpy()

    synthetic_df = pd.DataFrame(
        synthetic,
        columns=feature_names
    )

    print(f"Dataset            : {dataset_name}")
    print(f"Original Samples   : {len(original_dataframe)}")
    print(f"Synthetic Samples  : {len(synthetic_df)}")
    print(f"Features           : {synthetic_df.shape[1]}")

    print("=" * 70)

    return synthetic_df

In [179]:
# ============================================================
# Generate Synthetic Data for All Datasets
# ============================================================

synthetic_datasets = {}

for dataset_name, state in training_state.items():

    synthetic_df = generate_synthetic_data(

        generator=state["generator"],

        dataset_name=dataset_name,

        original_dataframe=datasets[dataset_name],

        latent_dim=LATENT_DIM,

        device=DEVICE

    )

    synthetic_datasets[dataset_name] = synthetic_df

Generating Synthetic Data : Adult Income


ValueError: Shape of passed values is (32510, 14), indices imply (32510, 15)

In [181]:
import torch

# ============================================================
# Verify Generator Output Dimension
# ============================================================

for dataset_name, state in training_state.items():

    print("=" * 60)
    print(dataset_name)

    # Correctly access the input dimension from tensor_metadata
    dataset_input_dim = tensor_metadata[dataset_name]["input_dimension"]

    print("Dataset columns :", dataset_input_dim)

    noise = torch.randn(2, LATENT_DIM).to(DEVICE)

    with torch.no_grad():
        fake = state["generator"](noise)

    print("Generator output :", fake.shape)

    print("=" * 60)

Adult Income
Dataset columns : 14
Generator output : torch.Size([2, 14])
Bank Marketing
Dataset columns : 16
Generator output : torch.Size([2, 16])
Breast Cancer
Dataset columns : 31
Generator output : torch.Size([2, 31])


In [183]:
for dataset_name, state in training_state.items():

    print("=" * 60)
    print(dataset_name)

    # Access the dataframe from the global 'datasets' dictionary
    df = datasets[dataset_name]

    print("Stored dataframe shape:")
    print(df.shape)

    print()

    print("Columns")
    print(df.columns.tolist())

    print("=" * 60)

Adult Income
Stored dataframe shape:
(32510, 15)

Columns
['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income']
Bank Marketing
Stored dataframe shape:
(45211, 17)

Columns
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']
Breast Cancer
Stored dataframe shape:
(569, 32)

Columns
['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave_points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave_points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst'

In [163]:
# ============================================================
# 6.8.9 Generate Synthetic Data
# ============================================================

def generate_synthetic_data(
    generator,
    dataset_name,
    original_dataframe,
    latent_dim,
    device,
    num_samples=None
):
    """
    Generate synthetic tabular data using a trained Generator.

    Parameters
    ----------
    generator : torch.nn.Module
        Trained generator.

    dataset_name : str
        Dataset name.

    original_dataframe : pandas.DataFrame
        Original processed dataset.

    latent_dim : int
        Latent vector dimension.

    device : torch.device

    num_samples : int, optional
        Number of synthetic records.
        Default = original dataset size.

    Returns
    -------
    pandas.DataFrame
    """

    generator.eval()

    if num_samples is None:
        num_samples = len(original_dataframe)

    feature_names = original_dataframe.columns.tolist()

    print("=" * 70)
    print(f"Generating Synthetic Data : {dataset_name}")
    print("=" * 70)

    with torch.no_grad():

        noise = torch.randn(
            num_samples,
            latent_dim,
            device=device
        )

        synthetic = generator(noise)

        synthetic = synthetic.cpu().numpy()

    synthetic_df = pd.DataFrame(
        synthetic,
        columns=feature_names
    )

    print(f"Dataset            : {dataset_name}")
    print(f"Original Samples   : {len(original_dataframe)}")
    print(f"Synthetic Samples  : {len(synthetic_df)}")
    print(f"Features           : {synthetic_df.shape[1]}")

    print("=" * 70)

    return synthetic_df

In [164]:
# ============================================================
# 6.8.10 Save Synthetic Datasets
# ============================================================

import os
import pandas as pd
from datetime import datetime


def save_synthetic_datasets(
    synthetic_datasets,
    project_dir,
    model_name="DPGAN"
):
    """
    Save all generated synthetic datasets.

    Parameters
    ----------
    synthetic_datasets : dict
        Dictionary of synthetic DataFrames.

    project_dir : str
        Root project directory.

    model_name : str
        Name of synthetic data generator.

    Returns
    -------
    dict
        Saved file paths.
    """

    save_directory = os.path.join(
        project_dir,
        "datasets",
        "synthetic",
        model_name
    )

    os.makedirs(
        save_directory,
        exist_ok=True
    )

    saved_files = {}

    summary = []

    print("=" * 80)
    print("Saving Synthetic Datasets")
    print("=" * 80)

    for dataset_name, synthetic_df in synthetic_datasets.items():

        filename = f"{dataset_name}_synthetic.csv"

        filepath = os.path.join(
            save_directory,
            filename
        )

        synthetic_df.to_csv(
            filepath,
            index=False
        )

        saved_files[dataset_name] = filepath

        summary.append({

            "Dataset": dataset_name,

            "Rows": synthetic_df.shape[0],

            "Columns": synthetic_df.shape[1],

            "File": filename

        })

        print(f"✓ {dataset_name}")
        print(f"  Shape : {synthetic_df.shape}")
        print(f"  Saved : {filepath}")
        print("-" * 60)

    # --------------------------------------------------------
    # Save Summary
    # --------------------------------------------------------

    summary_df = pd.DataFrame(summary)

    summary_path = os.path.join(
        save_directory,
        "synthetic_dataset_summary.csv"
    )

    summary_df.to_csv(
        summary_path,
        index=False
    )

    # --------------------------------------------------------
    # Save Metadata
    # --------------------------------------------------------

    metadata = {

        "Model": model_name,

        "Generated_On":

            datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            ),

        "Datasets":

            list(synthetic_datasets.keys()),

        "Total_Datasets":

            len(synthetic_datasets)

    }

    metadata_path = os.path.join(
        save_directory,
        "generation_metadata.json"
    )

    import json

    with open(
        metadata_path,
        "w"
    ) as file:

        json.dump(
            metadata,
            file,
            indent=4
        )

    print("=" * 80)
    print("All Synthetic Datasets Saved Successfully")
    print("=" * 80)

    return {

        "directory": save_directory,

        "saved_files": saved_files,

        "summary": summary_df,

        "summary_path": summary_path,

        "metadata_path": metadata_path

    }

In [166]:
# ============================================================
# 6.8.11 Generate All Synthetic Datasets
# ============================================================

synthetic_datasets = {}

for dataset_name, state in training_state.items():

    print(f"Processing dataset: {dataset_name}")

    # Get the trained generator and original dataframe
    generator = state["generator"]
    original_df = datasets[dataset_name]

    synthetic_df = generate_synthetic_data(
        generator=generator,
        dataset_name=dataset_name,
        original_dataframe=original_df,
        latent_dim=LATENT_DIM,
        device=DEVICE,
        num_samples=len(original_df)
    )

    synthetic_datasets[dataset_name] = synthetic_df

print("\nAll synthetic datasets generated successfully.")

# ============================================================
# Save All Synthetic Datasets
# ============================================================

saved_results = save_synthetic_datasets(

    synthetic_datasets=synthetic_datasets,

    project_dir=PROJECT_DIR,

    model_name="DPGAN"

)

Processing dataset: Adult Income
Generating Synthetic Data : Adult Income


ValueError: Shape of passed values is (32510, 14), indices imply (32510, 15)